# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/HassanNawaz14/FlyRank-ML-Internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## Setup — data, connection, and Week-4/5 baseline + model prep

*Reused verbatim from Weeks 3-5: DuckDB connection, cached aggregation, `feature_df` build, and the Week-4 baseline rule. No new decisions are made here.*

In [ ]:
import os
os.makedirs('skills/hunting-leakage-and-validating', exist_ok=True)
os.makedirs('skills/flyrank/flyrank-data', exist_ok=True)
!wget -q -O "skills/hunting-leakage-and-validating/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/hunting-leakage-and-validating/SKILL.md"
!wget -q -O "skills/flyrank/flyrank-data/SKILL.md" "https://raw.githubusercontent.com/HassanNawaz14/FlyRank-ML-Internship/refs/heads/main/skills/flyrank/flyrank-data/SKILL.md"
print("skills loaded")


In [ ]:
import duckdb, pandas as pd, os

HF_TOKEN = os.getenv('HF_TOKEN')
if HF_TOKEN is None:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get('HF_TOKEN')
    except Exception:
        pass
if HF_TOKEN is None:
    from getpass import getpass
    HF_TOKEN = getpass('Enter your Hugging Face token: ')

con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
TABLES = {
    'dim_clients': f"read_parquet('{REL}/dim_clients.parquet')",
    'dim_content': f"read_parquet('{REL}/dim_content.parquet')",
    'fact_daily':  f"read_parquet('{REL}/fact_content_daily_performance/**/*.parquet')",
}
print(con.execute(f"SELECT COUNT(*) FROM {TABLES['dim_clients']}").fetchone())


In [ ]:
# Materialize the expensive aggregation ONCE -- fact_daily has ~79M rows
# spread across many remote parquet files. Do not repeat this step.
max_date = pd.to_datetime(con.execute(f"SELECT MAX(report_date) FROM {TABLES['fact_daily']}").fetchone()[0])
last30_start = max_date - pd.Timedelta(days=30)
prev30_start = max_date - pd.Timedelta(days=60)

con.execute(f"""
CREATE OR REPLACE TABLE agg_cached AS
SELECT client_hash_id, content_hash_id,
    SUM(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_impressions ELSE 0 END) AS imp_last30,
    SUM(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_clicks ELSE 0 END) AS clk_last30,
    AVG(CASE WHEN report_date > DATE '{last30_start.date()}' THEN gsc_avg_position END) AS pos_last30,
    SUM(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_impressions ELSE 0 END) AS imp_prev30,
    SUM(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_clicks ELSE 0 END) AS clk_prev30,
    AVG(CASE WHEN report_date > DATE '{prev30_start.date()}' AND report_date <= DATE '{last30_start.date()}' THEN gsc_avg_position END) AS pos_prev30
FROM {TABLES['fact_daily']}
GROUP BY client_hash_id, content_hash_id
""")
print(con.execute("SELECT COUNT(*) FROM agg_cached").fetchone())


In [ ]:
query = f"""
SELECT
    a.*,
    cl.access_profile, cl.gsc_data_start, cl.ga4_data_start,
    dc.word_count, dc.char_count, dc.content_type, dc.main_intent,
    dc.content_updated_date,
    DATE_DIFF('day', dc.content_updated_date, DATE '{max_date.date()}') AS days_since_last_update
FROM agg_cached a
JOIN {TABLES['dim_clients']} cl ON a.client_hash_id = cl.client_hash_id
LEFT JOIN {TABLES['dim_content']} dc ON a.content_hash_id = dc.content_hash_id
WHERE a.imp_prev30 >= 100
"""
feature_df = con.execute(query).fetchdf()
feature_df['ctr_prev30'] = feature_df['clk_prev30'] / feature_df['imp_prev30'] * 100
feature_df['is_declining_label'] = feature_df['imp_last30'] < 0.8 * feature_df['imp_prev30']
print(feature_df.shape)


In [ ]:
import numpy as np

bins = [0, 90, 180, np.inf]
labels = ['<=90', '91-180', '181+']
feature_df['staleness_bucket'] = pd.cut(feature_df['days_since_last_update'], bins=bins, labels=labels, right=True)

pos_bins = [-np.inf, 3, 10, 20, 50, np.inf]
pos_labels = ['top_3', 'page_1', 'striking', 'page_3_5', 'deep']
feature_df['position_tier_prev'] = pd.cut(feature_df['pos_prev30'], bins=pos_bins, labels=pos_labels, right=True)

median_ctr_by_tier = feature_df.groupby('position_tier_prev', observed=True)['ctr_prev30'].median().rename('median_ctr_prev30').reset_index()
feature_df = feature_df.merge(median_ctr_by_tier, on='position_tier_prev', how='left')

feature_df['weak_ctr_flag'] = (
    (feature_df['ctr_prev30'] == 0) |
    ((feature_df['median_ctr_prev30'] > 0) & (feature_df['ctr_prev30'] < feature_df['median_ctr_prev30']))
)
feature_df['stale_flag'] = (feature_df['days_since_last_update'] > 90).astype(int)
feature_df['baseline_score'] = feature_df['stale_flag'] + feature_df['weak_ctr_flag'].astype(int)
feature_df['baseline_pred'] = (feature_df['baseline_score'] >= 1).astype(int)

print(feature_df['baseline_pred'].value_counts())


## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**Finding A — "What Predicts Growth?" (ML Appendix, logistic regression, 71% holdout accuracy).**
This is the direct parallel to my own lane, so it's the natural first pick.
- *Where does the label come from?* Built from the paper's own `trend_direction` definition — but their threshold is **>10% / -10%** for up/down (per the "How to Read This Paper" page), while my project's threshold (from `data-dictionary.md`) is **>20% / -20%**. Same underlying concept, different operational cutoff — the code cell below checks how much that difference actually matters on my own data.
- *Does the validation design carry the claim?* The Methodology page states "Logistic Regression (80/20 split)" but never says whether that split is grouped by brand or randomly by page. With 57 brands in the portfolio, pages from the same brand likely share templates and publishing patterns — exactly the leakage risk my own Week 2 split design was built to avoid. This is a fair, evidence-grounded question to raise, not a claim the paper is wrong.

**Finding B — "The Freshness Multiplier" (Finding #4 — the 57x impressions / 3.2x health headline, repeated as Playbook Priority #1).**
- *Where does the label come from?* A growth/decline proxy similar to `trend_direction`, applied specifically within the 365+ day age tier.
- *Does the validation design carry the claim?* The paper itself admits, in Finding #8, that the 365+ age tier carries "strong survivor bias" and should not be used as a headline decay proof point — but that caveat is never cross-referenced back to Finding #4's headline number, and Finding #4 doesn't disclose its own sample size the way the neighboring freshness-ratio table does (which openly shows the 361+ bucket rests on just 1 declining page). Constructive question: does the 57x number rest on that same small, survivor-biased sample flagged two pages later?

In [ ]:
# Check how much the paper's >10%/-10% threshold vs. my project's >20%/-20%
# threshold actually changes the classification, on my own real data.
trend_pct = (feature_df['imp_last30'] - feature_df['imp_prev30']) / feature_df['imp_prev30'].replace(0, pd.NA)

paper_down = (trend_pct < -0.10).sum()
project_down = (trend_pct < -0.20).sum()
n = feature_df.shape[0]

print(f"Total rows: {n}")
print(f"Flagged 'down' under paper's >10% threshold:   {paper_down} ({paper_down/n:.1%})")
print(f"Flagged 'down' under project's >20% threshold: {project_down} ({project_down/n:.1%})")
print(f"Rows that flip between the two thresholds: {paper_down - project_down}")


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

**"After" = Week 5's actual approach**: `GroupShuffleSplit` on `client_hash_id`, so no client appears in both train and test. This is the honest split, reused verbatim from Week 5.

**"Before" = the naive default**: a plain random `train_test_split` on the same rows, same features, same `test_size`/`random_state` — the split most people reach for by default, which lets rows from the same client leak between train and test.

Note: the baseline rule itself can't leak — it's a fixed hand-written rule never fit to training data — so this before/after comparison is really about Logistic Regression and Random Forest specifically.

In [ ]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

feature_cols_numeric = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30',
                        'days_since_last_update', 'word_count', 'char_count']
feature_cols_categorical = ['content_type', 'main_intent']

model_df = feature_df.copy()
model_df[feature_cols_numeric] = model_df[feature_cols_numeric].fillna(model_df[feature_cols_numeric].median())
model_df[feature_cols_categorical] = model_df[feature_cols_categorical].fillna('unknown')

X = pd.get_dummies(model_df[feature_cols_numeric + feature_cols_categorical],
                    columns=feature_cols_categorical, drop_first=True)
y = model_df['is_declining_label'].astype(int)
groups = model_df['client_hash_id']

def train_and_score(X_train, X_test, y_train, y_test, label):
    lr = LogisticRegression(max_iter=1000, random_state=42)
    lr.fit(X_train, y_train)
    lr_pred = lr.predict(X_test)

    rf = RandomForestClassifier(n_estimators=200, random_state=42, n_jobs=-1)
    rf.fit(X_train, y_train)
    rf_pred = rf.predict(X_test)

    return pd.DataFrame({
        'split': [label, label],
        'model': ['Logistic Regression', 'Random Forest'],
        'precision': [precision_score(y_test, lr_pred), precision_score(y_test, rf_pred)],
        'recall':    [recall_score(y_test, lr_pred), recall_score(y_test, rf_pred)],
        'f1':        [f1_score(y_test, lr_pred), f1_score(y_test, rf_pred)],
    })

# AFTER (honest): grouped split by client_hash_id -- same as Week 5
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx_g, test_idx_g = next(gss.split(X, y, groups=groups))
results_grouped = train_and_score(
    X.iloc[train_idx_g], X.iloc[test_idx_g], y.iloc[train_idx_g], y.iloc[test_idx_g],
    'AFTER (grouped by client)'
)

# BEFORE (naive): plain random split, no grouping
X_train_r, X_test_r, y_train_r, y_test_r = train_test_split(X, y, test_size=0.2, random_state=42)
results_random = train_and_score(X_train_r, X_test_r, y_train_r, y_test_r, 'BEFORE (random, no grouping)')

comparison = pd.concat([results_random, results_grouped], ignore_index=True)
print(comparison.to_string(index=False))


**Fill this in after running the cell above — do not guess the numbers:**

- Did the random ("before") split score higher than the grouped ("after") split on F1? `[FILL IN]`
- By how much did the random split inflate the numbers, for each model? `[FILL IN]`
- One or two sentences on what this proves about skipping a grouped split: `[FILL IN]`

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Final feature set used in Week 5 / Section 2 above:** `imp_prev30`, `clk_prev30`, `pos_prev30`, `ctr_prev30`, `days_since_last_update`, `word_count`, `char_count`, `content_type`, `main_intent`.

**Explicitly excluded, restated:** `imp_last30`, `clk_last30`, `pos_last30`, `trend_direction`, `trend_pct` — these overlap the window used to define `is_declining_label`, direct leakage.

**Also excluded, and worth restating deliberately:** the `fact_query_90d` columns (`rare_share`, `top_query_share`, `visible_queries`, `anon_share`) flagged as leakage-risk in Week 3 (that table's fixed 90-day window overlaps the label period) were never included in the final Week 5 model at all — this was a deliberate exclusion carried through consistently, not something forgotten.

In [ ]:
# Same leakage hunt as Week 3: correlate the final feature set against the
# label, and separately confirm the excluded columns correlate suspiciously
# higher (as expected, since they overlap the label window).
safe_features = ['imp_prev30', 'clk_prev30', 'pos_prev30', 'ctr_prev30', 'days_since_last_update']
excluded_features = ['imp_last30', 'clk_last30', 'pos_last30']

print("--- Correlation with is_declining_label: SAFE features (expect low/moderate) ---")
for col in safe_features:
    corr = feature_df[[col]].assign(label=feature_df['is_declining_label'].astype(int))[col].corr(feature_df['is_declining_label'].astype(int))
    print(f"{col}: {corr:.4f}")

print("\n--- Correlation with is_declining_label: EXCLUDED features (expect much higher) ---")
for col in excluded_features:
    corr = feature_df[col].corr(feature_df['is_declining_label'].astype(int))
    print(f"{col}: {corr:.4f}")


**Fill this in after running the cell above:**

- Do the excluded, label-window columns show visibly higher correlation with `is_declining_label` than the safe features? `[FILL IN]` — this is the expected pattern and confirms why they're excluded.
- Do any of the "safe" features show a surprisingly high correlation that deserves a second look? `[FILL IN]`

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

Two real candidates from earlier weeks, pick whichever felt boldest to write at the time:

**Option A (Week 4):** "Staleness past 90 days is a real, observed signal."
→ Rewritten: *In this dataset's sample, the observed decline rate rose across staleness buckets (roughly 62% at ≤90 days, 73% at 91-180 days, 79% at 181+ days). This is a directional association in the data reviewed, not a guarantee that any individual stale page is declining, nor proof that staleness causes decline.*

**Option B (Week 5):** "`ctr_prev30` dominates" (from the Logistic Regression coefficient ranking).
→ Rewritten: *`ctr_prev30` had the largest coefficient magnitude in the Logistic Regression fit on this feature set. Because the numeric features weren't scaled before fitting, this ranking is directional evidence at best, not a confirmed measure of relative importance — Random Forest's permutation importance ranked different features highest, and that disagreement itself is part of the honest finding.*

`[FILL IN: state which option was chosen, or substitute a different sentence pulled from your own Week 1-5 write-ups, and confirm the rewrite is grounded in the real printed numbers from that week rather than restated confidently.]`

In [ ]:
# Re-print the real numbers backing whichever claim was chosen above,
# so the rewrite in the markdown cell is grounded in actual output.

# Option A support: staleness bucket table (from the baseline cell above)
staleness_summary = feature_df.groupby('staleness_bucket', observed=True).agg(
    n=('content_hash_id', 'count'),
    is_declining_rate=('is_declining_label', lambda x: x.mean() * 100)
).reset_index()
print("--- Option A support: staleness vs decline rate ---")
print(staleness_summary.to_string())


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it (Sections 2, 3, and 4 have `[FILL IN]` placeholders — complete these with real printed numbers before checking this box)
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.